In [1]:
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics import f1_score, normalized_mutual_info_score
import warnings

warnings.filterwarnings('ignore')

def compute_pairwise_se_distance(X, Y):

    XX = np.sum(X**2, axis=1, keepdims=True)
    YY = np.sum(Y**2, axis=1, keepdims=True).T
    XY = np.dot(X, Y.T)
    return XX + YY - 2*XY

def compute_centroids(opinions, labels, K):
    centroids = np.zeros((K, opinions.shape[1]))
    for k in range(K):
        mask = (labels == k)
        if np.any(mask):
            centroids[k] = opinions[mask].mean(axis=0)
        else:
            centroids[k] = np.zeros(opinions.shape[1])
    return centroids

def avg_f1_score(true_labels, pred_labels, true_K=None, pred_K=None):
    true_labels = np.asarray(true_labels)
    pred_labels = np.asarray(pred_labels)
    if true_K is None:
        true_K = len(np.unique(true_labels))
    if pred_K is None:
        pred_K = len(np.unique(pred_labels))

    true_onehot = np.eye(true_K)[true_labels]
    pred_onehot = np.eye(pred_K)[pred_labels]

    sum_pred = 0.0
    for p in range(pred_K):
        f1_scores = []
        for t in range(true_K):
            f1 = f1_score(true_onehot[:, t], pred_onehot[:, p], zero_division=0)
            f1_scores.append(f1)
        sum_pred += max(f1_scores)

    sum_true = 0.0
    for t in range(true_K):
        f1_scores = []
        for p in range(pred_K):
            f1 = f1_score(true_onehot[:, t], pred_onehot[:, p], zero_division=0)
            f1_scores.append(f1)
        sum_true += max(f1_scores)

    avgf1 = (sum_pred / (2 * pred_K)) + (sum_true / (2 * true_K))
    return avgf1

# ---------------------------- GK-means 算法类 ----------------------------
class GKMeans:
    def __init__(self, K=2, d=64, beta=None, rho=0.5, epsilon=None,
                 max_iter_phase1=100, max_iter_phase3=30, random_state=42):
        self.K = K
        self.d = d
        self.beta = beta if beta is not None else d
        self.rho = rho
        self.epsilon = epsilon if epsilon is not None else d * 0.001
        self.max_iter_phase1 = max_iter_phase1
        self.max_iter_phase3 = max_iter_phase3
        self.random_state = random_state
        np.random.seed(random_state)

    def fit(self, G):
        n = G.number_of_nodes()
        m = G.number_of_edges()
        nodes = list(G.nodes())
        adj = nx.to_numpy_array(G, nodelist=nodes, dtype=int)
        degrees = adj.sum(axis=1).astype(int)

        neighbors = [set(G.neighbors(i)) for i in nodes]

   
        LN_matrix = np.zeros((n, n))
        for i in range(n):
            for j in neighbors[i]:
                if i >= j:  # 无向图去重
                    continue
                common = len(neighbors[i] & neighbors[j])
                ln_val = (2 * common) / (degrees[i] * degrees[j]) if degrees[i]*degrees[j] > 0 else 0
                LN_matrix[i, j] = ln_val
                LN_matrix[j, i] = ln_val

        # ---------- Phase 1: Opinion Dynamics ----------
        opinions = np.random.rand(n, self.d)

        for t in range(self.max_iter_phase1):
            opinions_new = np.zeros_like(opinions)
            max_change = 0.0
            for i in range(n):
                dist_to_i = np.sum((opinions[list(neighbors[i])] - opinions[i])**2, axis=1)
                threshold = self.beta * (self.rho ** t)
                valid_mask = dist_to_i <= threshold
                valid_neighbors = [list(neighbors[i])[idx] for idx, v in enumerate(valid_mask) if v]

                if len(valid_neighbors) == 0:
                    opinions_new[i] = opinions[i]
                    alpha_tilde = 1.0
                else:
                    ln_vals = np.array([LN_matrix[i, j] for j in valid_neighbors])
                    w_sum = ln_vals.sum()
                    w = ln_vals / w_sum if w_sum > 0 else np.ones(len(valid_neighbors)) / len(valid_neighbors)
                    alpha_tilde = np.random.rand()
                    weighted_opinions = np.sum(w[:, np.newaxis] * opinions[valid_neighbors], axis=0)
                    opinions_new[i] = alpha_tilde * opinions[i] + (1 - alpha_tilde) * weighted_opinions

                change = np.sum((opinions_new[i] - opinions[i])**2)
                if change > max_change:
                    max_change = change

            opinions = opinions_new
            if max_change < self.epsilon:
                break

        # ---------- Phase 2: Opinion Leader Identification ----------
        R = np.zeros(n)
        for i in range(n):
            for j in neighbors[i]:
                union = len(neighbors[i] | neighbors[j])
                jc = len(neighbors[i] & neighbors[j]) / union if union > 0 else 0
                dist_se = np.sum((opinions[i] - opinions[j])**2)
                R[i] += jc * np.exp(-dist_se)

        sorted_nodes = sorted(range(n), key=lambda x: R[x], reverse=True)

        labels = -np.ones(n, dtype=int)
        visited = np.zeros(n, dtype=bool)
        leader_count = 0
        for i in sorted_nodes:
            if leader_count >= self.K:
                break
            if not visited[i]:
                labels[i] = leader_count
                visited[i] = True
                for j in neighbors[i]:
                    visited[j] = True
                leader_count += 1

        unassigned = np.where(labels == -1)[0]
        if len(unassigned) > 0:
            leader_centroids = []
            for k in range(self.K):
                mask = (labels == k)
                if np.any(mask):
                    leader_centroids.append(opinions[mask].mean(axis=0))
                else:
                    leader_centroids.append(np.zeros(self.d))
            leader_centroids = np.array(leader_centroids)
            for i in unassigned:
                dists = np.sum((leader_centroids - opinions[i])**2, axis=1)
                labels[i] = np.argmin(dists)

        # ---------- Phase 3: Finding Locally Pareto Optimality ----------
        labels = labels.astype(int)
        for tau in range(self.max_iter_phase3):
            theta = np.zeros(self.K)
            for k in range(self.K):
                mask = (labels == k)
                theta[k] = degrees[mask].sum() / (2 * m) if m > 0 else 0

            eta = np.zeros((n, self.K))
            for i in range(n):
                if degrees[i] == 0:
                    continue
                neigh_labels = labels[list(neighbors[i])]
                for k in range(self.K):
                    eta[i, k] = np.sum(neigh_labels == k) / degrees[i]

            new_labels = labels.copy()
            for i in range(n):
                current_label = labels[i]
                feasible_set = []
                for p in range(self.K):
                    if p == current_label:
                        feasible_set.append(p)
                        continue
                        
                    theta_p_new = theta[p] + degrees[i] / (2 * m)
                    theta_q_new = theta[current_label] - degrees[i] / (2 * m)

                    def L_i(label, theta_val, eta_val):
                        return - (degrees[i] / (2 * m)) * (eta_val - theta_val)

                    L_i_p = L_i(p, theta_p_new, eta[i, p])
                    L_i_q = L_i(current_label, theta_q_new, eta[i, current_label])

                    if 2 * (L_i_p - L_i_q) <= 0:
                        feasible_set.append(p)

                if not feasible_set:
                    feasible_set = [current_label]

                best_p = current_label
                best_dist = np.inf
                for p in feasible_set:
                    temp_labels = labels.copy()
                    temp_labels[i] = p
                    mask = (temp_labels == p)
                    if np.any(mask):
                        centroid = opinions[mask].mean(axis=0)
                    else:
                        centroid = np.zeros(self.d)
                    dist = np.sum((opinions[i] - centroid)**2)
                    if dist < best_dist:
                        best_dist = dist
                        best_p = p
                new_labels[i] = best_p

            if np.array_equal(new_labels, labels):
                break
            labels = new_labels

        return labels


if __name__ == "__main__":
    print(">>> 正在初始化模型与环境配置...")


    all_edges = pd.read_csv('FaceBK.edges', header=None, sep=r'\s+')
    all_edges = all_edges - 1
    row = list(all_edges.iloc[:, 0])
    col = list(all_edges.iloc[:, 1])
    
   
    n = max(max(row), max(col)) + 1
    
    print(f"[*] 成功载入网络拓扑，解析有效边：{len(row)} 条")

    W = np.zeros((n, n))
    for i in range(len(row)):
        W[row[i]][col[i]] = 1
        W[col[i]][row[i]] = 1  
        
    G = nx.from_numpy_array(W)

  
    all_community = pd.read_csv('FaceBK.gt', header=None)
    TRUE_COMMUNITY = np.zeros(n, dtype=int)
    all_nodes_number = 0
    
  
    true_K = len(all_community.iloc[:, 0])
    
    for i in range(true_K):
    
        community_node = str(all_community.iloc[:, 0][i]).strip().split(' ')
        community_node = [x for x in community_node if x]
        all_nodes_number += len(community_node)
        
        nodes_label = [int(x) for x in community_node]
        nodes_label = np.array(nodes_label) - 1
        
       
        valid_nodes = [x for x in nodes_label if x < n]
        TRUE_COMMUNITY[valid_nodes] = i

    print(f"[*] 节点规模 n = {n}，聚类数 K = {true_K}\n")

 
    K = true_K
    d = 64               
    beta = d
    rho = 0.5
    epsilon = d * 0.001
    max_iter_phase1 = 100
    max_iter_phase3 = 30
    n_runs = 20         

    all_pred_labels = []
    all_avgf1 = []
    all_nmi = []


    for run in range(n_runs):
        gk = GKMeans(K=K, d=d, beta=beta, rho=rho, epsilon=epsilon,
                     max_iter_phase1=max_iter_phase1, max_iter_phase3=max_iter_phase3,
                     random_state=run)
        pred = gk.fit(G)
        all_pred_labels.append(pred)

        avgf1 = avg_f1_score(TRUE_COMMUNITY, pred, true_K=true_K, pred_K=K)
        nmi = normalized_mutual_info_score(TRUE_COMMUNITY, pred)
        all_avgf1.append(avgf1)
        all_nmi.append(nmi)
        print(f"  [-] 第 {run+1:02d}/20 轮运算完毕 -> AvgF1: {avgf1:.4f}, NMI: {nmi:.4f}")

    # 4. 数据存档
    df_results = pd.DataFrame(np.array(all_pred_labels).T, columns=[f"Run_{i+1}" for i in range(n_runs)])
    df_results.to_excel("FaceBK_clustering_results_20runs.xlsx", index=False)
    
    metrics_df = pd.DataFrame({
        "Run": np.arange(1, n_runs+1),
        "AvgF1": all_avgf1,
        "NMI": all_nmi
    })
    
    avg_avgf1 = np.mean(all_avgf1)
    avg_nmi = np.mean(all_nmi)

    mean_row = pd.DataFrame({"Run": ["Average"], "AvgF1": [avg_avgf1], "NMI": [avg_nmi]})
    metrics_df = pd.concat([metrics_df, mean_row], ignore_index=True)
    metrics_df.to_excel("FaceBK_metrics_20runs.xlsx", index=False)
    
    print("\n================== 任务总结 ==================")
    print(">>> 标签预测存储至: FaceBK_clustering_results_20runs.xlsx")
    print(">>> 验证指标存储至: FaceBK_metrics_20runs.xlsx")
    print(f"*** 最终测评: Mean AvgF1 = {avg_avgf1:.4f} | Mean NMI = {avg_nmi:.4f} ***")

>>> 正在初始化模型与环境配置...
[*] 成功载入网络拓扑，解析有效边：15783 条
[*] 解析社区状态完毕，确认节点规模 n = 793，网络真实聚类数 K = 2

>>> 进入核心算法，将开展 20 轮蒙特卡洛聚类任务...
  [-] 第 01/20 轮运算完毕 -> AvgF1: 0.9667, NMI: 0.7948
  [-] 第 02/20 轮运算完毕 -> AvgF1: 0.9705, NMI: 0.8120
  [-] 第 03/20 轮运算完毕 -> AvgF1: 0.9654, NMI: 0.7892
  [-] 第 04/20 轮运算完毕 -> AvgF1: 0.9654, NMI: 0.7892
  [-] 第 05/20 轮运算完毕 -> AvgF1: 0.9602, NMI: 0.7643
  [-] 第 06/20 轮运算完毕 -> AvgF1: 0.9641, NMI: 0.7838
  [-] 第 07/20 轮运算完毕 -> AvgF1: 0.9654, NMI: 0.7892
  [-] 第 08/20 轮运算完毕 -> AvgF1: 0.9641, NMI: 0.7838
  [-] 第 09/20 轮运算完毕 -> AvgF1: 0.9705, NMI: 0.8120
  [-] 第 10/20 轮运算完毕 -> AvgF1: 0.9641, NMI: 0.7838
  [-] 第 11/20 轮运算完毕 -> AvgF1: 0.9679, NMI: 0.8004
  [-] 第 12/20 轮运算完毕 -> AvgF1: 0.9537, NMI: 0.7428
  [-] 第 13/20 轮运算完毕 -> AvgF1: 0.9589, NMI: 0.7627
  [-] 第 14/20 轮运算完毕 -> AvgF1: 0.9692, NMI: 0.8061
  [-] 第 15/20 轮运算完毕 -> AvgF1: 0.9641, NMI: 0.7807
  [-] 第 16/20 轮运算完毕 -> AvgF1: 0.9563, NMI: 0.7487
  [-] 第 17/20 轮运算完毕 -> AvgF1: 0.9603, NMI: 0.7630
  [-] 第 18/20 轮运算完毕 -> AvgF1: